# FlyRank Capstone — Structured Content Archetype Clustering
**Lane:** Structured Content Archetype Clustering
**Author:** Ali (Mycodelab-lab)

Groups pages into six performance archetypes — protect, improve, rewrite, merge, prune, monitor —
using unsupervised clustering over safe search signals, then produces a ranked action table.

Set `SOURCE = "real"` once your Hugging Face token is exported in this environment
(`HUGGING_FACE_HUB_TOKEN`). Ships as `"synthetic"` so the notebook runs end-to-end with no token.

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib nbformat

In [ ]:
import sys
sys.path.append("../../scripts")  # pipeline lives in scripts/, this notebook in work/notebooks/

SOURCE = "synthetic"  # flip to "real" once your HF token is active

In [ ]:
from archetype_pipeline import (
    load_data, engineer_features, grouped_holdout, fit_clusters, score_holdout,
    map_clusters_to_archetypes, build_action_table, NUMERIC_FEATURES
)

df = load_data(SOURCE)
truth = df.pop("_archetype_truth") if "_archetype_truth" in df.columns else None
df = engineer_features(df)
df.head()

## Validation split
Held out by **client**, not by row — so no client's pages appear in both train and test.
`trend_direction` / `trend_pct` are never used as features (see `scripts/archetype_pipeline.py` header).

In [ ]:
train_df, holdout_df = grouped_holdout(df)
print(f"train pages: {len(train_df)}  |  holdout pages: {len(holdout_df)}  |  "
      f"train clients: {train_df['client_id'].nunique()}  |  holdout clients: {holdout_df['client_id'].nunique()}")

## Fit clusters (k chosen by silhouette score)

In [ ]:
fit = fit_clusters(train_df)
print("best k:", fit["best_k"])
fit["silhouette_scores"]

In [ ]:
train_df = train_df.copy()
train_df["cluster"] = fit["train_labels"]

holdout_labels, holdout_silhouette = score_holdout(fit, holdout_df)
holdout_df = holdout_df.copy()
holdout_df["cluster"] = holdout_labels
print("holdout silhouette:", holdout_silhouette)

## Map clusters to named archetypes and build the ranked action table

In [ ]:
import pandas as pd

full = pd.concat([train_df, holdout_df])
mapping = map_clusters_to_archetypes(fit, full)
mapping

In [ ]:
action_table = build_action_table(full, mapping)
action_table.head(15)

## Save output for the paper

In [ ]:
action_table.to_csv("../../data/clustered_pages.csv", index=False)
action_table["archetype"].value_counts()